[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaushikreddym/climdata/blob/main/docs/examples/wrapper_workflow.ipynb)

In [ ]:
# Uncomment to install climdata in Google Colab or other environments
# !pip install climdata

# ClimData Tutorial
This notebook demonstrates usage of the `ClimData` class for climate data extraction, extreme index computation, and workflow management.
Includes examples for point-based and box-based extraction, variable exploration, and error handling.

## 1️⃣ Imports

In [ ]:
from climdata import ClimData
import pandas as pd
import xarray as xr

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s",
    force=True,
)

## 2️⃣ Explore available datasets

In [ ]:
extractor = ClimData()
datasets = extractor.get_datasets()
print(datasets)

## 3️⃣ Explore variables for a dataset

In [ ]:
variables = extractor.get_variables('w5e5')
print(variables)

In [ ]:
# Explore the CMIP6 catalogue
import climdata
extractor_CMIP = climdata.CMIP(extractor.cfg)
print("Available Experiments (experiment_id)")
print("="*60)
print(extractor_CMIP.get_experiment_ids())
print("="*60)
print("Available CMIP6 Models (source_id)")
print("="*60)
print(extractor_CMIP.get_source_ids('ssp245'))
print("="*60)
print("Variables")
print("="*60)
print(extractor_CMIP.get_variables(experiment_id='ssp245',source_id='ACCESS-CM2'))
print("="*60)

## 4️⃣ Explore metadata for a variable

In [ ]:
varinfo = extractor.get_varinfo('rlds')
print(varinfo)

## 5️⃣ Explore available workflow actions

In [ ]:
actions = extractor.get_actions()
print(actions.keys())

In [ ]:
indices = extractor.get_indices(['tasmin', 'tasmax'])
print(indices.keys())

impute_methods = extractor.get_impute_methods()
print(impute_methods.keys())

## 6️⃣ Point extraction workflow

In [ ]:
import json

# -----------------------------
# Step 1: Define the area of interest (AOI)
# -----------------------------
# The AOI is a single point. In GeoJSON format, the coordinates are [longitude, latitude].
geojson = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {},
      "geometry": {
        "coordinates": [
          24.246667038198012,  # longitude
          12.891982026993958   # latitude
        ],
        "type": "Point"
      }
    }
  ]
}


# -----------------------------
# Step 2: Define configuration overrides
# -----------------------------
# Overrides are strings used by Hydra to modify default configurations at runtime.
overrides = [
    "dataset=cmip",  # Choose the CMIP6 dataset for extraction
    f"aoi='{json.dumps(geojson)}'",  # Set the AOI as the point defined above
    "time_range.start_date=2004-01-01",  # Start date for data extraction
    "time_range.end_date=2014-12-31",    # End date for data extraction
    "variables=[tasmin,tasmax,pr]",       # Variables to extract: min/max temp and precipitation
    "data_dir=./data",    # Local directory to store raw/intermediate files
    "index=tn10p",  # Climate extreme index to calculate
    "impute=BRITS"
]

# -----------------------------
# Step 3: Define the workflow sequence
# -----------------------------
seq = ["extract", "impute", "calc_index", "to_nc"]

# -----------------------------
# Step 4: Initialize the ClimData extractor
# -----------------------------
extractor = ClimData(overrides=overrides)

# -----------------------------
# Step 5: Run the Multi-Step workflow
# -----------------------------
result = extractor.run_workflow(
    actions=seq,
)

In [ ]:
import json

# -----------------------------
# Define the area of interest (AOI)
# -----------------------------
# This AOI is a single point with latitude 12.891982026993958 and longitude 24.246667038198012
geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {},
            "geometry": {
                "coordinates": [24.246667038198012, 12.891982026993958],
                "type": "Point"
            }
        }
    ]
}

# -----------------------------
# Define configuration overrides
# -----------------------------
# These strings override the default hydra config at runtime
overrides = [
    "dataset=mswx",  # Select the MSWX dataset for extraction
    f"aoi='{json.dumps(geojson)}'",  # Set AOI as the point defined above
    "time_range.start_date=2014-12-01",  # Start date of extraction
    "time_range.end_date=2014-12-31",    # End date of extraction
    "variables=[tasmin,tasmax,pr]",       # Variables to extract: min/max temperature & precipitation
    "data_dir=./data",    # Local directory to store downloaded/intermediate files
    # Optional Google service account if needed for MSWX access
    # "dsinfo.mswx.params.google_service_account=./.climdata_conf/service.json",
    "index=tn10p",  # Extreme climate index to calculate
]

# -----------------------------
# Initialize the ClimData extractor
# -----------------------------
# This loads the configuration with overrides and prepares the object
extractor = ClimData(overrides=overrides)

# -----------------------------
# Extract climate data
# -----------------------------
# Returns an xarray.Dataset for the selected variables, AOI, and time range
ds = extractor.extract()

# -----------------------------
# Compute the climate index
# -----------------------------
# Takes the extracted dataset and calculates the extreme index "tn10p"
# Returns a new xarray.Dataset containing only the index
ds_index = extractor.calc_index(ds)

# -----------------------------
# Convert the index dataset to a long-form pandas DataFrame
# -----------------------------
# Each row corresponds to a time, lat, lon, and variable (here just "tn10p")
df_index = extractor.to_dataframe(ds_index)

# -----------------------------
# Save the DataFrame to CSV
# -----------------------------
# This will write the index values to "index.csv" in the current working directory
extractor.to_csv(df_index, filename="index.csv")

### Output filenames

In [ ]:
print(extractor.current_filename)

## 7️⃣ Box extraction workflow

In [ ]:
box_overrides = [
    "dataset=mswx",  # Select the MSWX dataset for extraction
    "region=europe", # Select the region
    "variables=[tasmin,tasmax]",
    "time_range.start_date=2014-12-01",  # Start date of extraction
    "time_range.end_date=2014-12-31",    # End date of extraction
    "data_dir=./data",    # Local directory to store downloaded/intermediate files
]

extractor_box = ClimData(overrides=box_overrides)
result_box = extractor_box.run_workflow(actions=["extract", "to_csv"])

## 8️⃣ Compute extreme index only

In [ ]:
lat_berlin, lon_berlin = [52.5,13.4]
idx_overrides = [
    "dataset=mswx",  # Select the MSWX dataset for extraction
    f"lat={lat_berlin}", # Select the region
    f"lon={lon_berlin}",
    "variables=[tasmin,tasmax]",
    "time_range.start_date=2014-12-01",  # Start date of extraction
    "time_range.end_date=2014-12-31",    # End date of extraction
    "data_dir=./data",    # Local directory to store downloaded/intermediate files
    "index=heat_wave_max_length"
]


extractor_idx = ClimData(overrides=idx_overrides)
result_idx = extractor_idx.run_workflow(actions=["extract", "calc_index", "to_csv"])
result_idx.dataframe.head()

## 9️⃣ Error examples

In [ ]:
try:
    bad_ex = ClimData()
    bad_ex.run_workflow(actions=["calc_index"])
except Exception as e:
    print("Error:", e)

try:
    bad_ex = ClimData()
    bad_ex.run_workflow(actions=["to_csv"])
except Exception as e:
    print("Error:", e)

try:
    bad_ex = ClimData()
    bad_ex.run_workflow(actions=["upload_netcdf"])
except Exception as e:
    print("Error:", e)